In [ ]:
import os
import sys
from pathlib import Path

import torch
from miditok import REMI

# make `src` importable when the kernel's cwd is playground/
REPO = Path.cwd().resolve().parent if Path.cwd().name == "playground" else Path.cwd().resolve()
sys.path.insert(0, str(REPO))

from src.model.model import MusicTransformer

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps') # metal for Apple Silicon
else:
    device = torch.device('cpu')

print("Using device:", device)

In [ ]:
data_root = REPO / "data"
ckpt_path = Path(os.environ.get("MUSIC_LLM_CKPT", data_root / "checkpoints" / "best.pt"))
tokenizer_path = data_root / "tokenized" / "config" / "tokenizer.json"
val_tok_folder = data_root / "tokenized" / "val"
output_folder = data_root / "output"

In [ ]:
# Load checkpoint; torch.compile'd training runs save keys with an "_orig_mod." prefix (known issue #3)
ckpt = torch.load(ckpt_path, map_location=device)
state = {k.removeprefix("_orig_mod."): v for k, v in ckpt["model"].items()}

model = MusicTransformer(vocab_size=ckpt["vocab_size"], **ckpt["hyperparams"]).to(device)
model.load_state_dict(state)
print("Loaded", ckpt_path.name, "from epoch", ckpt["epoch"], "with val loss", round(ckpt["best_val_loss"], 4))
print("Hyperparameters:", ckpt["hyperparams"])

In [ ]:
# Prompt: the first PROMPT_LEN tokens of one validation song
import json

PROMPT_LEN = 64
song = sorted(val_tok_folder.glob("*.json"))[0]
ids = json.loads(song.read_text())["ids"]
prompt = torch.tensor(ids[:PROMPT_LEN], dtype=torch.long, device=device).unsqueeze(0) # (1, PROMPT_LEN)
print(song.stem, "-", len(ids), "tokens, prompting with", PROMPT_LEN)

In [ ]:
torch.manual_seed(0)
out = model.generate(prompt, max_new_tokens=1024, temperature=0.9, top_k=40, top_p=0.95).cpu()
print(out.shape)

In [ ]:
tokenizer = REMI(params=tokenizer_path)
print("Is trained", tokenizer.is_trained)

output_folder.mkdir(exist_ok=True)
prompt_midi = tokenizer.decode(out[0, :PROMPT_LEN].tolist())
gen_midi = tokenizer.decode(out[0].tolist())
prompt_midi.dump_midi(output_folder / f"{song.stem}_prompt.mid")
gen_midi.dump_midi(output_folder / f"{song.stem}_generated.mid")
print("Wrote", f"{song.stem}_prompt.mid", "and", f"{song.stem}_generated.mid", "to", output_folder)

In [ ]:
# Grammar check: share of generated tokens that break REMI's note structure (Program, Pitch, Velocity, Duration)
from miditok import TokSequence

def grammar_error_rate(ids):
    seq = TokSequence(ids=ids, are_ids_encoded=True)
    tokenizer.decode_token_ids(seq)
    tokenizer.complete_sequence(seq)
    return tokenizer.tokens_errors(seq)

print("prompt    error rate:", round(grammar_error_rate(out[0, :PROMPT_LEN].tolist()), 3))
print("generated error rate:", round(grammar_error_rate(out[0, PROMPT_LEN:].tolist()), 3))
print("notes in prompt / full output:", sum(len(t.notes) for t in prompt_midi.tracks), "/", sum(len(t.notes) for t in gen_midi.tracks))